In [2]:
from os import rename

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
import importlib
import config
importlib.reload(config)
import pandas_market_calendars as mcal

from config import RAW_SENT_C,SENT_TIMESTAMP_CLEANED

In [3]:
"""
Align sentiment timestamps to NYSE trading days.

Rules (all judged in America/New_York local time):
    1. Trading day, before 16:00:00        -> unchanged
    2. Trading day, at/after 16:00:00      -> next trading day at 00:01:00
    3. Non-trading day (weekend / holiday) -> next trading day at 00:01:00

Input : parquet at RAW_SENT_C, timestamp column 'Timestamp' (UTC)
Output: parquet at SENT_TIMESTAMP_CLEANED with
        'Timestamp_utc' (original, renamed) and
        'Timestamp'     (cleaned, tz-aware America/New_York)
"""

NY_TZ = "America/New_York"
CLOSE_HOUR = 16            # 16:00:00 exactly counts as after the close
CALENDAR_BUFFER_DAYS = 10  # so the last rows always have a "next trading day"


def clean_sentiment_timestamps(df: pd.DataFrame) -> pd.DataFrame:
    """Rename 'Timestamp' to 'Timestamp_utc' and build the cleaned 'Timestamp' column."""
    df = df.rename(columns={"Timestamp": "Timestamp_utc"})

    timestamp_utc = df["Timestamp_utc"]
    if timestamp_utc.dt.tz is None:
        # Data is known to be UTC but stored naive - make that explicit
        timestamp_utc = timestamp_utc.dt.tz_localize("UTC")
        df["Timestamp_utc"] = timestamp_utc

    timestamp_ny = timestamp_utc.dt.tz_convert(NY_TZ)

    # --- NYSE trading days as sorted naive midnights ---
    nyse_calendar = mcal.get_calendar("NYSE")
    trading_days = nyse_calendar.valid_days(
        start_date=timestamp_ny.min().date(),
        end_date=timestamp_ny.max().date() + pd.Timedelta(days=CALENDAR_BUFFER_DAYS),
    )
    trading_day_values = trading_days.tz_localize(None).values  # datetime64[ns], sorted

    # --- classify every row (fully vectorised) ---
    ny_date = timestamp_ny.dt.normalize().dt.tz_localize(None)  # naive NY-local midnight
    is_trading_day = ny_date.isin(trading_day_values)
    at_or_after_close = timestamp_ny.dt.hour >= CLOSE_HOUR
    needs_move = ~is_trading_day | at_or_after_close

    timestamp_clean = timestamp_ny.copy()

    if needs_move.any():
        # First trading day STRICTLY after each row's NY date
        move_dates = ny_date[needs_move].values
        next_idx = np.searchsorted(trading_day_values, move_dates, side="right")
        if next_idx.max() >= len(trading_day_values):
            raise ValueError(
                "Calendar range too short - increase CALENDAR_BUFFER_DAYS."
            )
        next_trading_day = trading_day_values[next_idx]

        # 00:01 never falls in a DST transition (those occur at 02:00),
        # so tz_localize is always unambiguous here
        moved_timestamps = (
            pd.DatetimeIndex(next_trading_day) + pd.Timedelta(minutes=1)
        ).tz_localize(NY_TZ)

        timestamp_clean.loc[needs_move] = moved_timestamps

    df["Timestamp"] = timestamp_clean

    # --- audit summary ---
    n_total = len(df)
    n_non_trading = int((~is_trading_day).sum())
    n_after_close = int((is_trading_day & at_or_after_close).sum())
    print(f"Rows total:                      {n_total:,}")
    print(f"Moved (weekend / holiday):       {n_non_trading:,}")
    print(f"Moved (trading day >= 16:00 NY): {n_after_close:,}")
    print(f"Unchanged:                       {n_total - n_non_trading - n_after_close:,}")

    return df


if __name__ == "__main__":
    sentiment_df = pd.read_parquet(RAW_SENT_C)
    sentiment_df = clean_sentiment_timestamps(sentiment_df)
    sentiment_df.to_parquet(SENT_TIMESTAMP_CLEANED, index=False)
    print(f"Saved: {SENT_TIMESTAMP_CLEANED}")

Rows total:                      3,546,807
Moved (weekend / holiday):       186,597
Moved (trading day >= 16:00 NY): 1,091,617
Unchanged:                       2,268,593
Saved: T:\CGATES\Alexandria\raw_sentiment_d.parquet


In [4]:
sentiment_df.head(100)

,StoryID,Timestamp_utc,Ticker,Country,Isin,Sentiment,Confidence,Novelty,Topic,Relevance,HeadlineOnly,AutoMessage,Source,MarketImpactScore,Prob_POS,Prob_NTR,Prob_NEG,Timestamp
0,ZDM4Q1VWMVNTUUlFQVFRREFRZFFYdz09,2018-01-01 00:10:06.842000+00:00,BA,USA,US0970231058,0,0.726263,1,AA@ALEX,1.000000,F,F,DJN,-0.742605,0.094962,0.817509,0.087529,2018-01-02 00:01:00-05:00
1,ZDM4Q1VWMVNTUUlFQVFRREFRZFFXZz09,2018-01-01 00:10:06.845000+00:00,BRKB,USA,US0846707026,0,0.965210,1,AA@ALEX,1.000000,F,F,DJN,0.016196,0.014441,0.976806,0.008753,2018-01-02 00:01:00-05:00
2,ZDM4Q1VWMVNTUUlFQVFRREFRZFFXdz09,2018-01-01 00:10:06.845000+00:00,CME,USA,US12572Q1058,0,0.334720,1,AA@ALEX,0.500000,F,F,DJN,-0.926168,0.430337,0.556480,0.013184,2018-01-02 00:01:00-05:00
3,ZDM4Q1VWMVNTUUlFQVFRREFRZFRYdz09,2018-01-01 00:20:06.797000+00:00,M,USA,US55616P1049,0,0.826251,1,AA@ALEX,0.333333,F,F,DJN,1.065464,0.100773,0.884167,0.015060,2018-01-02 00:01:00-05:00
4,ZDM4Q1VWMVNTUUlFQVFRREFRZFVYZz09,2018-01-01 01:36:08.596000+00:00,BA,USA,US0970231058,-1,0.763068,1,AA@ALEX,0.500000,F,F,TPC,0.939084,0.029239,0.128715,0.842045,2018-01-02 00:01:00-05:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,ZDM4Q1VWMWRTQUVIQVFRREFRTlZWQT09,2018-01-01 12:12:21.829000+00:00,TMUS,USA,US8725901040,0,0.984163,1,"AA@ALEX,AA@INS",0.076923,F,F,TPC,-1.010352,0.001145,0.989442,0.009413,2018-01-02 00:01:00-05:00
96,ZDM4Q1VWMWRTQUVIQVFRREFRTlZWUT09,2018-01-01 12:12:31.847000+00:00,FND,USA,US3397501012,0,0.996977,1,"AA@ALEX,AA@INS",0.055556,F,F,TPC,-1.010352,0.000779,0.997985,0.001236,2018-01-02 00:01:00-05:00
97,ZDM4Q1VWMWRTQUVIQVFRREFRTlZWUT09,2018-01-01 12:12:31.847000+00:00,GPS,USA,US3647601083,0,0.996977,1,"AA@ALEX,AA@INS",0.055556,F,F,TPC,-1.010352,0.000779,0.997985,0.001236,2018-01-02 00:01:00-05:00
98,ZDM4Q1VWMWRTQUVIQVFRREFRTlZWUT09,2018-01-01 12:12:31.847000+00:00,KSS,USA,US5002551043,0,0.996977,1,"AA@ALEX,AA@INS",0.055556,F,F,TPC,-1.010352,0.000779,0.997985,0.001236,2018-01-02 00:01:00-05:00
